# Implement PPO for RLHF

**Difficulty**: 🔴 Hard

**Companies**: Anthropic, OpenAI, DeepMind, Meta

---

### Problem Statement

Proximal Policy Optimization (PPO) is the workhorse algorithm behind RLHF (Reinforcement Learning from Human Feedback), used to train models like ChatGPT and Claude. In the RLHF pipeline, a reward model (trained on human preferences) provides scalar rewards, and PPO optimizes the policy to maximize those rewards while staying close to a reference policy.

PPO uses a clipped surrogate objective to prevent the policy from changing too drastically in a single update, along with a value function baseline to reduce variance, and a KL penalty to keep the policy close to the reference.

Your task is to implement the full PPO-RLHF pipeline: policy model, value model, reward model, GAE computation, and the PPO clipped objective with KL penalty.

---

### Requirements

1. **PolicyModel** — Generates sequences and outputs action logits.
2. **ValueModel** — Estimates the value (expected future reward) of states.
3. **RewardModel** — Scores sequences (frozen, pretrained).
4. **compute_gae** — Generalized Advantage Estimation.
5. **ppo_step** — The PPO clipped objective update with KL penalty.

---

### Constraints

- ✅ Use a simple task: generate sequences that sum to a target value.
- ✅ Include KL penalty against a reference policy.
- ✅ Average reward should increase over training.
- ✅ KL divergence from reference should stay bounded.
- ❌ Do **not** use any existing RL/RLHF libraries (trl, stable-baselines, etc.).

---

<details>
  <summary>💡 Hint</summary>

  - GAE: A_t = delta_t + (gamma * lam) * delta_{t+1} + ... where delta_t = r_t + gamma * V(s_{t+1}) - V(s_t).
  - PPO clipped objective: L = min(ratio * A, clip(ratio, 1-eps, 1+eps) * A) where ratio = pi(a|s) / pi_old(a|s).
  - For KL penalty: KL = mean(log_pi - log_ref) across actions. Add -beta * KL to the reward or subtract from the objective.
  - Process: (1) generate sequences from policy, (2) score with reward model, (3) compute GAE advantages, (4) do multiple PPO epochs on the collected batch.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

In [ ]:
# Task: generate sequences of discrete tokens.
# Reward = -|sum(tokens) - target|  (closer to target = higher reward)
VOCAB_SIZE = 10
SEQ_LEN = 8
TARGET_SUM = 36  # target sum of token values


class PolicyModel(nn.Module):
    """Policy network: takes a state (partial sequence) and outputs logits over vocab."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)  # +1 for BOS token
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """Returns logits for each position: (batch, seq_len, vocab_size)."""
        x = self.embedding(input_ids)
        h, _ = self.rnn(x)
        return self.head(h)


class ValueModel(nn.Module):
    """Value network: estimates expected future reward from current state."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        """Returns value estimates: (batch, seq_len, 1)."""
        x = self.embedding(input_ids)
        h, _ = self.rnn(x)
        return self.head(h).squeeze(-1)  # (batch, seq_len)


class RewardModel(nn.Module):
    """Reward model: scores a complete sequence. Reward = -|sum(tokens) - target|."""
    def __init__(self, target_sum=TARGET_SUM):
        super().__init__()
        self.target_sum = target_sum

    def forward(self, sequences):
        """sequences: (batch, seq_len) of token IDs. Returns (batch,) rewards."""
        sums = sequences.float().sum(dim=-1)
        return -torch.abs(sums - self.target_sum)


torch.manual_seed(42)
policy = PolicyModel()
value_model = ValueModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

print("Policy params:", sum(p.numel() for p in policy.parameters()))
print("Value params:", sum(p.numel() for p in value_model.parameters()))

In [ ]:
def generate_sequences(policy: nn.Module, batch_size: int, seq_len: int,
                        vocab_size: int) -> tuple:
    """
    Generate sequences autoregressively from the policy.
    
    Returns:
        sequences: (batch, seq_len) generated token IDs
        log_probs: (batch, seq_len) log probability of each token under policy
        all_input_ids: (batch, seq_len) input IDs at each step (for value model)
    """
    # TODO: Start with BOS token (vocab_size) for each batch element
    # TODO: At each step, get logits from policy, sample an action, record log prob
    # TODO: Append sampled token and continue
    # TODO: Return sequences, log_probs, and input sequences for value estimation
    ...


def compute_gae(rewards: torch.Tensor, values: torch.Tensor,
                gamma: float = 0.99, lam: float = 0.95) -> torch.Tensor:
    """
    Compute Generalized Advantage Estimation.
    
    Args:
        rewards: (batch, seq_len) per-step rewards (only last step is nonzero here).
        values: (batch, seq_len) value estimates from value model.
        gamma: Discount factor.
        lam: GAE lambda for bias-variance tradeoff.
    
    Returns:
        advantages: (batch, seq_len) advantage estimates.
    """
    # TODO: Compute TD residuals: delta_t = r_t + gamma * V(t+1) - V(t)
    # TODO: Compute GAE by iterating backwards:
    #       A_t = delta_t + gamma * lam * A_{t+1}
    ...


def ppo_step(policy: nn.Module, value_model: nn.Module, ref_policy: nn.Module,
             old_log_probs: torch.Tensor, input_ids: torch.Tensor,
             actions: torch.Tensor, advantages: torch.Tensor,
             returns: torch.Tensor, clip_epsilon: float = 0.2,
             kl_beta: float = 0.01) -> dict:
    """
    Compute PPO clipped objective with KL penalty.
    
    Args:
        policy: Current policy model.
        value_model: Value function model.
        ref_policy: Reference (frozen) policy for KL penalty.
        old_log_probs: Log probs from the policy that generated the data.
        input_ids: Input token IDs at each step. (batch, seq_len)
        actions: Actions (tokens) taken. (batch, seq_len)
        advantages: GAE advantages. (batch, seq_len)
        returns: advantages + values (targets for value function). (batch, seq_len)
        clip_epsilon: PPO clipping parameter.
        kl_beta: KL penalty coefficient.
    
    Returns:
        Dict with 'policy_loss', 'value_loss', 'kl_div', 'total_loss'.
    """
    # TODO: Get current policy log probs for the actions taken
    # TODO: Get reference policy log probs for KL computation
    # TODO: Compute ratio = exp(new_log_probs - old_log_probs)
    # TODO: Compute clipped and unclipped objective, take the min
    # TODO: Compute value loss = MSE(values, returns)
    # TODO: Compute KL divergence = mean(new_log_probs - ref_log_probs)
    # TODO: Total loss = -policy_loss + 0.5 * value_loss + kl_beta * kl_div
    ...

In [ ]:
# === Validation ===
torch.manual_seed(42)

policy = PolicyModel()
value_model = ValueModel()
ref_policy = copy.deepcopy(policy)
for p in ref_policy.parameters():
    p.requires_grad = False
reward_model = RewardModel()

policy_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
value_optimizer = torch.optim.Adam(value_model.parameters(), lr=1e-3)

batch_size = 64
num_iterations = 30
ppo_epochs = 4

reward_history = []
kl_history = []

print("=== PPO Training ===")
for iteration in range(num_iterations):
    # 1. Generate sequences from current policy
    policy.eval()
    with torch.no_grad():
        sequences, old_log_probs, input_ids = generate_sequences(
            policy, batch_size, SEQ_LEN, VOCAB_SIZE)
        rewards_final = reward_model(sequences)  # (batch,)
        values = value_model(input_ids)  # (batch, seq_len)

    # Spread reward to last step only
    step_rewards = torch.zeros(batch_size, SEQ_LEN)
    step_rewards[:, -1] = rewards_final

    # 2. Compute GAE advantages
    advantages = compute_gae(step_rewards, values.detach())
    returns = advantages + values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # 3. PPO update epochs
    policy.train()
    value_model.train()
    for _ in range(ppo_epochs):
        loss_dict = ppo_step(policy, value_model, ref_policy,
                             old_log_probs, input_ids, sequences,
                             advantages, returns)

        policy_optimizer.zero_grad()
        value_optimizer.zero_grad()
        loss_dict['total_loss'].backward()
        policy_optimizer.step()
        value_optimizer.step()

    avg_reward = rewards_final.mean().item()
    avg_kl = loss_dict['kl_div'].item()
    reward_history.append(avg_reward)
    kl_history.append(avg_kl)

    if iteration % 5 == 0:
        print(f"Iter {iteration:3d} | Reward: {avg_reward:.2f} | KL: {avg_kl:.4f}")

# Test 1: Reward should increase
print("\n=== Reward Increase Test ===")
early_reward = sum(reward_history[:5]) / 5
late_reward = sum(reward_history[-5:]) / 5
print(f"Early avg reward: {early_reward:.2f}")
print(f"Late avg reward:  {late_reward:.2f}")
assert late_reward > early_reward, "Average reward should increase over training!"
print("PASSED\n")

# Test 2: KL should stay bounded
print("=== KL Bounded Test ===")
max_kl = max(kl_history)
print(f"Max KL divergence: {max_kl:.4f}")
assert max_kl < 50.0, "KL divergence should stay bounded!"
print("PASSED\n")

print("All tests passed!")